# Tiled Matrix Multiplication — PMPP Ch. 5

**Runtime → Change runtime type → T4 GPU** before running.

Ch. 5 introduces the key memory optimization: **tiling with phases**.  
Each block loads small tiles of M and N into on-chip `__shared__` memory, computes partial sums, then moves to the next tile (next phase).  
With `TILE_WIDTH = N`, global memory traffic drops by **N×**.

In [ ]:
!pip install nvcc4jupyter -q
%load_ext nvcc4jupyter

## Recap: Naive kernel — O(N³) global memory reads

Every thread reads its own full row of M and column of N from **global memory (DRAM)**, the slowest memory tier.  
For a 1000×1000 multiply that's **2 billion** DRAM accesses — the CGMA ratio is only **1.0**.

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH 256

__global__ void matmul_naive(float *M, float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < width && col < width) {
        float sum = 0.0f;
        for (int k = 0; k < width; k++)
            sum += M[row * width + k] * N[k * width + col];
        P[row * width + col] = sum;
    }
}

int main() {
    size_t size = WIDTH * WIDTH * sizeof(float);
    float *h_M = (float*)malloc(size), *h_N = (float*)malloc(size), *h_P = (float*)malloc(size);

    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i*WIDTH+j] = (i==j) ? 1.0f : 0.0f;
            h_N[i*WIDTH+j] = (i==j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size); cudaMalloc(&d_N, size); cudaMalloc(&d_P, size);
    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(16,16), grid(WIDTH/16, WIDTH/16);
    matmul_naive<<<grid,block>>>(d_M, d_N, d_P, WIDTH);
    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++)
            if (fabsf(h_P[i*WIDTH+j] - (i==j ? 1.0f : 0.0f)) > 1e-3f) ok = 0;
    printf("Naive:  %s\n", ok ? "correct" : "WRONG");
    printf("Global reads: %lld\n", (long long)WIDTH*WIDTH*WIDTH*2);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
}

## Ch. 5 Core Idea — Tiling with Phases

### Memory hierarchy
| Memory | Scope | Latency | Size |
|--------|-------|---------|------|
| Registers | per thread | 1 cycle | tiny (~255 regs/thread) |
| **Shared (`__shared__`)** | **per block** | **~5 cycles** | **48 KB/SM** |
| Global (DRAM) | all threads | ~500 cycles | GBs |

### The tiling strategy (Fig 5.4 in the book)

The output matrix P is split into `TILE_WIDTH × TILE_WIDTH` output tiles, one tile per block.  
Each block computes its tile in **phases**:

```
for phase t in 0 .. (WIDTH/TILE_WIDTH - 1):
    1. All threads in block collaboratively load tile t of M  →  Mds[TILE_WIDTH][TILE_WIDTH]
    2. All threads in block collaboratively load tile t of N  →  Nds[TILE_WIDTH][TILE_WIDTH]
    3. __syncthreads()   ← barrier: every thread must finish loading before anyone computes
    4. Each thread accumulates its dot-product partial sum from Mds and Nds
    5. __syncthreads()   ← barrier: done computing before next phase overwrites shared mem
```

Each element of M/N is loaded from DRAM **once per phase it participates in**, then reused `TILE_WIDTH` times from shared memory.  
→ **CGMA ratio improves from 1.0 → TILE_WIDTH** (e.g. 16× with a 16×16 tile).

## Tiled kernel — annotated phase loop

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH      256
#define TILE_WIDTH  16   // try 8 or 32 — CGMA ratio scales with this

__global__ void matmul_tiled(float *M, float *N, float *P, int width) {
    // On-chip shared memory tiles — one per block, reused across phases
    __shared__ float Mds[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Nds[TILE_WIDTH][TILE_WIDTH];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE_WIDTH + ty;   // global row this thread owns
    int col = blockIdx.x * TILE_WIDTH + tx;   // global col this thread owns

    float sum = 0.0f;   // lives in a register — private per thread

    // --- Phase loop: step one tile-width at a time across the shared dimension ---
    int numPhases = width / TILE_WIDTH;
    for (int t = 0; t < numPhases; t++) {

        // PHASE LOAD: each thread loads exactly one element into shared memory
        // Thread (ty,tx) fetches column (t*TILE_WIDTH + tx) of M for its row
        Mds[ty][tx] = M[row * width + (t * TILE_WIDTH + tx)];
        // Thread (ty,tx) fetches row (t*TILE_WIDTH + ty) of N for its column
        Nds[ty][tx] = N[(t * TILE_WIDTH + ty) * width + col];

        // BARRIER 1: wait for all threads to finish loading before anyone reads
        // Without this, a fast thread could read Nds[k][tx] before a slow
        // thread has written it (race condition → wrong result, see Ex 5.6.3)
        __syncthreads();

        // PHASE COMPUTE: accumulate partial dot-product from shared tiles
        for (int k = 0; k < TILE_WIDTH; k++)
            sum += Mds[ty][k] * Nds[k][tx];

        // BARRIER 2: wait for all threads to finish computing before the next
        // phase overwrites Mds/Nds (shared memory is reused each phase)
        __syncthreads();
    }

    if (row < width && col < width)
        P[row * width + col] = sum;
}

int main() {
    size_t size = WIDTH * WIDTH * sizeof(float);
    float *h_M = (float*)malloc(size), *h_N = (float*)malloc(size), *h_P = (float*)malloc(size);

    // Identity × Identity = Identity (easy to verify)
    for (int i = 0; i < WIDTH; i++)
        for (int j = 0; j < WIDTH; j++) {
            h_M[i*WIDTH+j] = (i==j) ? 1.0f : 0.0f;
            h_N[i*WIDTH+j] = (i==j) ? 1.0f : 0.0f;
        }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size); cudaMalloc(&d_N, size); cudaMalloc(&d_P, size);
    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    dim3 block(TILE_WIDTH, TILE_WIDTH);
    dim3 grid(WIDTH / TILE_WIDTH, WIDTH / TILE_WIDTH);
    matmul_tiled<<<grid, block>>>(d_M, d_N, d_P, WIDTH);
    cudaMemcpy(h_P, d_P, size, cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < WIDTH && ok; i++)
        for (int j = 0; j < WIDTH && ok; j++)
            if (fabsf(h_P[i*WIDTH+j] - (i==j ? 1.0f : 0.0f)) > 1e-3f) ok = 0;
    printf("Tiled:  %s\n", ok ? "correct" : "WRONG");
    printf("CGMA ratio: %d  (naive was 1, tiled is TILE_WIDTH)\n", TILE_WIDTH);
    printf("Global reads reduced by %dx vs naive\n", TILE_WIDTH);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
}

## Benchmark: naive vs tiled

`cudaEvent` timing directly on the GPU — more accurate than Python `time.time()`.

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define WIDTH      512
#define TILE_WIDTH  16

__global__ void matmul_naive(float *M, float *N, float *P, int width) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < width && col < width) {
        float s = 0.0f;
        for (int k = 0; k < width; k++)
            s += M[row*width+k] * N[k*width+col];
        P[row*width+col] = s;
    }
}

__global__ void matmul_tiled(float *M, float *N, float *P, int width) {
    __shared__ float Mds[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Nds[TILE_WIDTH][TILE_WIDTH];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE_WIDTH + ty;
    int col = blockIdx.x * TILE_WIDTH + tx;
    float sum = 0.0f;
    for (int t = 0; t < width / TILE_WIDTH; t++) {
        Mds[ty][tx] = M[row * width + (t * TILE_WIDTH + tx)];
        Nds[ty][tx] = N[(t * TILE_WIDTH + ty) * width + col];
        __syncthreads();
        for (int k = 0; k < TILE_WIDTH; k++)
            sum += Mds[ty][k] * Nds[k][tx];
        __syncthreads();
    }
    if (row < width && col < width)
        P[row*width+col] = sum;
}

float time_kernel(void (*launch)(float*,float*,float*,float*,int), float *dM, float *dN, float *dP, int w) {
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    launch(dM, dN, dP, NULL, w);   // dummy wrapper not used — see below
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float ms; cudaEventElapsedTime(&ms, start, stop);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return ms;
}

int main() {
    size_t size = (size_t)WIDTH * WIDTH * sizeof(float);
    float *h_M = (float*)malloc(size), *h_N = (float*)malloc(size), *h_P = (float*)malloc(size);
    for (int i = 0; i < WIDTH*WIDTH; i++) { h_M[i] = 1.0f; h_N[i] = 1.0f; }

    float *d_M, *d_N, *d_P;
    cudaMalloc(&d_M, size); cudaMalloc(&d_N, size); cudaMalloc(&d_P, size);
    cudaMemcpy(d_M, h_M, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_N, h_N, size, cudaMemcpyHostToDevice);

    cudaEvent_t s, e; float ms;
    dim3 block16(16,16), grid16(WIDTH/16, WIDTH/16);
    dim3 blockT(TILE_WIDTH,TILE_WIDTH), gridT(WIDTH/TILE_WIDTH, WIDTH/TILE_WIDTH);

    // --- naive ---
    cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    matmul_naive<<<grid16, block16>>>(d_M, d_N, d_P, WIDTH);
    cudaEventRecord(e); cudaEventSynchronize(e);
    cudaEventElapsedTime(&ms, s, e);
    printf("Naive  (%dx%d): %.2f ms\n", WIDTH, WIDTH, ms);
    cudaEventDestroy(s); cudaEventDestroy(e);

    // --- tiled ---
    cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    matmul_tiled<<<gridT, blockT>>>(d_M, d_N, d_P, WIDTH);
    cudaEventRecord(e); cudaEventSynchronize(e);
    cudaEventElapsedTime(&ms, s, e);
    printf("Tiled  (%dx%d, tile=%d): %.2f ms\n", WIDTH, WIDTH, TILE_WIDTH, ms);
    cudaEventDestroy(s); cudaEventDestroy(e);

    cudaFree(d_M); cudaFree(d_N); cudaFree(d_P);
    free(h_M); free(h_N); free(h_P);
}

## Ex 5.6.3 — What breaks without `__syncthreads()`?

The tiled kernel has **two** barriers:

1. **After loading** (`Mds`/`Nds` written) — without this, a fast thread starts computing from uninitialized shared memory (data race: read before write).
2. **After computing** (before next phase loads) — without this, a fast thread overwrites `Mds[ty][tx]` for phase `t+1` while a slow thread is still reading it for phase `t` (data race: write before read).

Both races produce silently wrong results — no crash, just bad numbers.